# TP - GAN Convolucional (DCGAN + WGAN)
### Redes Neuronales / Deep Learning

**Trabajo en grupos de 2 personas.** Pueden consultarse entre grupos.

**Defensa oral: 40 minutos por grupo.**

Esta es la **Notebook 3 de 3**. Vamos a implementar una GAN convolucional simplificada (estilo DCGAN) para generar imágenes de CIFAR-10, comparar su entrenamiento con una **WGAN** (Wasserstein GAN), y cerrar con una explicación grupal de una variante de GAN más avanzada.

## Objetivos
- Implementar Generador y Discriminador convolucionales.
- Entender y programar el entrenamiento adversarial (alternado G/D).
- Trackear el entrenamiento con TensorBoard y MLflow.
- Comparar la estabilidad de una GAN estándar vs. una WGAN.
- Investigar y explicar una variante de GAN (CycleGAN, Pix2Pix, cGAN, StyleGAN, SRGAN o WGAN-GP).


## 0. Setup

In [ ]:
!pip install -q mlflow


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import mlflow

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Usando dispositivo:", device)


## 1. Datos: CIFAR-10

Para GANs conviene normalizar a `[-1, 1]` en vez de `[0, 1]`, porque el generador va a terminar con una activación `Tanh` (funciona mejor que `Sigmoid` para este caso).

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # lleva [0,1] a [-1,1]
])

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)

LATENT_DIM = 100

def denormalize(x):
    return (x * 0.5) + 0.5  # vuelve a [0,1] para visualizar


## 2. Arquitectura DCGAN

**Generador**: parte de un vector de ruido `z` de dimensión `LATENT_DIM` y lo "infla" hasta una imagen `3x32x32` con capas `ConvTranspose2d`.

- `Linear` de `LATENT_DIM` a `256*4*4`, reshape a `(256,4,4)`
- ConvTranspose 256→128, kernel 4, stride 2, padding 1 → `128x8x8` (+ BatchNorm + ReLU)
- ConvTranspose 128→64, kernel 4, stride 2, padding 1 → `64x16x16` (+ BatchNorm + ReLU)
- ConvTranspose 64→3, kernel 4, stride 2, padding 1 → `3x32x32` (+ **Tanh**)

### TODO 1: completar el Generador

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 4 * 4)
        # TODO: completar las 3 capas de ConvTranspose2d según la tabla de arriba
        # (BatchNorm2d va DESPUÉS de la conv y ANTES de la activación)
        self.deconv = nn.Sequential(
            # nn.ConvTranspose2d(...), nn.BatchNorm2d(...), nn.ReLU(),
            # nn.ConvTranspose2d(...), nn.BatchNorm2d(...), nn.ReLU(),
            # nn.ConvTranspose2d(...), nn.Tanh(),
        )

    def forward(self, z):
        x = self.fc(z)
        x = x.view(x.size(0), 256, 4, 4)
        x = self.deconv(x)
        return x


**Discriminador**: es como el encoder de las notebooks anteriores, pero termina en un solo número (logit) que indica si la imagen es real o generada.

- Conv 3→64, kernel 4, stride 2, padding 1 → `64x16x16` (+ LeakyReLU(0.2))
- Conv 64→128, kernel 4, stride 2, padding 1 → `128x8x8` (+ BatchNorm + LeakyReLU(0.2))
- Conv 128→256, kernel 4, stride 2, padding 1 → `256x4x4` (+ BatchNorm + LeakyReLU(0.2))
- Flatten + `Linear` a 1 (**sin sigmoid**: devolvemos el logit crudo, usamos `BCEWithLogitsLoss` que es más estable numéricamente)

### TODO 2: completar el Discriminador

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: completar las 3 capas convolucionales según la tabla de arriba
        self.conv = nn.Sequential(
            # nn.Conv2d(...), nn.LeakyReLU(0.2, inplace=True),
            # nn.Conv2d(...), nn.BatchNorm2d(...), nn.LeakyReLU(0.2, inplace=True),
            # nn.Conv2d(...), nn.BatchNorm2d(...), nn.LeakyReLU(0.2, inplace=True),
        )
        self.fc = nn.Linear(256 * 4 * 4, 1)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)  # logit crudo


In [ ]:
# Sanity check
_z = torch.randn(4, LATENT_DIM)
_g_check = Generator(LATENT_DIM)
_fake = _g_check(_z)
assert _fake.shape == (4, 3, 32, 32), f"Shape incorrecta del generador: {_fake.shape}"

_d_check = Discriminator()
_out = _d_check(_fake)
assert _out.shape == (4, 1), f"Shape incorrecta del discriminador: {_out.shape}"
print("Shapes OK. Generador:", _fake.shape, "| Discriminador:", _out.shape)


## 3. Entrenamiento adversarial (ya resuelto, explicado paso a paso)

En cada batch:

1. **Paso del Discriminador**: le mostramos imágenes reales (etiqueta 1) y falsas generadas por G (etiqueta 0), y actualizamos D para que las distinga mejor.
2. **Paso del Generador**: generamos imágenes falsas y le pedimos a D que las evalúe. Actualizamos G para que D las clasifique como reales (etiqueta 1) — G "engaña" a D.

Usamos `BCEWithLogitsLoss` y *label smoothing* (etiquetas reales = 0.9 en vez de 1.0) para estabilizar un poco el entrenamiento.

In [ ]:
%load_ext tensorboard


In [ ]:
def train_dcgan(generator, discriminator, loader, epochs, lr=2e-4, writer=None, run_name="dcgan"):
    generator.to(device)
    discriminator.to(device)
    criterion = nn.BCEWithLogitsLoss()
    opt_g = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = optim.Adam(discriminator.parameters(), lr=lr, betas=(0.5, 0.999))

    fixed_noise = torch.randn(16, LATENT_DIM).to(device)
    g_losses, d_losses = [], []
    step = 0

    for epoch in range(epochs):
        for real_imgs, _ in loader:
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)
            real_labels = torch.full((batch_size, 1), 0.9, device=device)  # label smoothing
            fake_labels = torch.zeros((batch_size, 1), device=device)

            # --- 1. Entrenar Discriminador ---
            opt_d.zero_grad()
            pred_real = discriminator(real_imgs)
            loss_d_real = criterion(pred_real, real_labels)

            noise = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = generator(noise)
            pred_fake = discriminator(fake_imgs.detach())  # detach: no propagamos a G acá
            loss_d_fake = criterion(pred_fake, fake_labels)

            loss_d = loss_d_real + loss_d_fake
            loss_d.backward()
            opt_d.step()

            # --- 2. Entrenar Generador ---
            opt_g.zero_grad()
            pred_fake_for_g = discriminator(fake_imgs)  # sin detach: sí propagamos a G
            loss_g = criterion(pred_fake_for_g, torch.ones((batch_size, 1), device=device))
            loss_g.backward()
            opt_g.step()

            g_losses.append(loss_g.item())
            d_losses.append(loss_d.item())
            if writer is not None:
                writer.add_scalar("loss/generator", loss_g.item(), step)
                writer.add_scalar("loss/discriminator", loss_d.item(), step)
            step += 1

        avg_g = sum(g_losses[-len(loader):]) / len(loader)
        avg_d = sum(d_losses[-len(loader):]) / len(loader)
        print(f"[{run_name}] Epoch {epoch+1}/{epochs} - loss_G: {avg_g:.4f} - loss_D: {avg_d:.4f}")
        mlflow.log_metric("loss_G", avg_g, step=epoch)
        mlflow.log_metric("loss_D", avg_d, step=epoch)

        if writer is not None:
            generator.eval()
            with torch.no_grad():
                samples = denormalize(generator(fixed_noise))
            writer.add_image("generadas", make_grid(samples, nrow=4), epoch)
            generator.train()

    return g_losses, d_losses


In [ ]:
EPOCHS_DCGAN = 40

generator = Generator(LATENT_DIM)
discriminator = Discriminator()
writer = SummaryWriter(log_dir="runs/dcgan")

mlflow.set_experiment("gan_cifar10")
with mlflow.start_run(run_name="dcgan"):
    mlflow.log_params({"latent_dim": LATENT_DIM, "epochs": EPOCHS_DCGAN, "batch_size": BATCH_SIZE, "lr": 2e-4})
    g_losses, d_losses = train_dcgan(generator, discriminator, train_loader, epochs=EPOCHS_DCGAN, writer=writer)
    mlflow.pytorch.log_model(generator, "generator")

writer.close()


In [ ]:
%tensorboard --logdir runs


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(g_losses, label="Generador")
plt.plot(d_losses, label="Discriminador")
plt.xlabel("Iteración")
plt.ylabel("Loss")
plt.title("Curvas de loss - DCGAN")
plt.legend()
plt.show()

generator.eval()
with torch.no_grad():
    samples = denormalize(generator(torch.randn(16, LATENT_DIM).to(device)))
plt.figure(figsize=(8, 8))
plt.imshow(make_grid(samples.cpu(), nrow=4).permute(1, 2, 0))
plt.axis("off")
plt.title("Imágenes generadas - DCGAN")
plt.show()


## 4. WGAN (Wasserstein GAN) — sección dada, para comparar

Las GAN estándar suelen ser inestables: el discriminador puede volverse "demasiado bueno" muy rápido, dejando al generador sin gradiente útil (loss saturada), o el generador puede colapsar generando siempre imágenes muy parecidas (**mode collapse**).

**WGAN** propone:
- Cambiar el discriminador por un **crítico** que no clasifica (no hay sigmoid ni BCE), sino que estima una distancia (Wasserstein) entre la distribución real y la generada. La loss es simplemente la diferencia de scores: `D(real) - D(fake)`.
- Entrenar el crítico varias veces (`n_critic`) por cada actualización del generador.
- **Weight clipping**: después de cada update del crítico, recortar sus pesos a un rango chico `[-c, c]`, para que la función que aprende sea "1-Lipschitz" (una condición matemática que requiere la teoría de WGAN — no hace falta que la demuestren, alcanza con entender que sin esto el entrenamiento se vuelve inestable).

Reusamos la misma arquitectura de Generador/Discriminador (el discriminador ahora se llama "crítico" y ya devuelve un logit crudo, así que no hay que tocarlo).

In [ ]:
def train_wgan(generator, critic, loader, epochs, lr=5e-5, n_critic=5, clip_value=0.01,
               writer=None, run_name="wgan"):
    generator.to(device)
    critic.to(device)
    opt_g = optim.RMSprop(generator.parameters(), lr=lr)
    opt_c = optim.RMSprop(critic.parameters(), lr=lr)

    fixed_noise = torch.randn(16, LATENT_DIM).to(device)
    g_losses, c_losses = [], []
    step = 0

    for epoch in range(epochs):
        for real_imgs, _ in loader:
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)

            # --- Entrenar el crítico n_critic veces ---
            for _ in range(n_critic):
                opt_c.zero_grad()
                noise = torch.randn(batch_size, LATENT_DIM, device=device)
                fake_imgs = generator(noise).detach()
                loss_c = -(torch.mean(critic(real_imgs)) - torch.mean(critic(fake_imgs)))
                loss_c.backward()
                opt_c.step()
                # Weight clipping: la clave de WGAN "simple"
                for p in critic.parameters():
                    p.data.clamp_(-clip_value, clip_value)

            # --- Entrenar el generador ---
            opt_g.zero_grad()
            noise = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = generator(noise)
            loss_g = -torch.mean(critic(fake_imgs))
            loss_g.backward()
            opt_g.step()

            g_losses.append(loss_g.item())
            c_losses.append(loss_c.item())
            if writer is not None:
                writer.add_scalar("loss/generator", loss_g.item(), step)
                writer.add_scalar("loss/critic", loss_c.item(), step)
            step += 1

        avg_g = sum(g_losses[-len(loader):]) / len(loader)
        avg_c = sum(c_losses[-len(loader):]) / len(loader)
        print(f"[{run_name}] Epoch {epoch+1}/{epochs} - loss_G: {avg_g:.4f} - loss_C: {avg_c:.4f}")
        mlflow.log_metric("loss_G", avg_g, step=epoch)
        mlflow.log_metric("loss_C", avg_c, step=epoch)

        if writer is not None:
            generator.eval()
            with torch.no_grad():
                samples = denormalize(generator(fixed_noise))
            writer.add_image("generadas", make_grid(samples, nrow=4), epoch)
            generator.train()

    return g_losses, c_losses


In [ ]:
EPOCHS_WGAN = 35

generator_w = Generator(LATENT_DIM)
critic_w = Discriminator()
writer_w = SummaryWriter(log_dir="runs/wgan")

mlflow.set_experiment("gan_cifar10")
with mlflow.start_run(run_name="wgan"):
    mlflow.log_params({"latent_dim": LATENT_DIM, "epochs": EPOCHS_WGAN, "batch_size": BATCH_SIZE,
                        "lr": 5e-5, "n_critic": 5, "clip_value": 0.01})
    g_losses_w, c_losses_w = train_wgan(generator_w, critic_w, train_loader, epochs=EPOCHS_WGAN, writer=writer_w)
    mlflow.pytorch.log_model(generator_w, "generator")

writer_w.close()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(g_losses, label="Generador (DCGAN)")
axes[0].plot(d_losses, label="Discriminador (DCGAN)")
axes[0].set_title("DCGAN")
axes[0].legend()

axes[1].plot(g_losses_w, label="Generador (WGAN)")
axes[1].plot(c_losses_w, label="Crítico (WGAN)")
axes[1].set_title("WGAN")
axes[1].legend()
plt.suptitle("Comparación de estabilidad: DCGAN vs WGAN")
plt.show()

generator_w.eval()
with torch.no_grad():
    samples_w = denormalize(generator_w(torch.randn(16, LATENT_DIM).to(device)))
plt.figure(figsize=(8, 8))
plt.imshow(make_grid(samples_w.cpu(), nrow=4).permute(1, 2, 0))
plt.axis("off")
plt.title("Imágenes generadas - WGAN")
plt.show()


## 5. Mode collapse e inestabilidad: qué mirar

Algunas señales para diagnosticar problemas en una GAN (para comentar en la defensa con lo que les haya pasado a ustedes):

- **Mode collapse**: las imágenes generadas se parecen mucho entre sí (poca diversidad), aunque el ruido de entrada sea distinto.
- **Loss del discriminador cayendo a ~0**: señal de que está "ganando" la carrera y el generador dejó de recibir gradiente útil.
- **Loss del generador creciendo sin parar**: parecido al punto anterior, desde el lado del generador.
- En WGAN, la loss del crítico (a diferencia de BCE) sí tiene una interpretación más directa como "distancia" entre distribuciones, y en teoría debería ser más informativa sobre el progreso real del entrenamiento.


## 6. Trabajo grupal final: variante de GAN

Cada grupo va a investigar y explicar (con un **diagrama simple**, a mano o hecho en cualquier herramienta) una variante de GAN distinta a las que vimos acá. La variante se asigna de forma **reproducible según el número de grupo** (así no hay que coordinarse a mano y no se pisan entre grupos).

Completen `NUMERO_DE_GRUPO` con el número que les asignó la cátedra y corran la celda.

In [ ]:
import random as _random

NUMERO_DE_GRUPO = 1  # <-- reemplazar por el número de grupo asignado por la cátedra

VARIANTES_GAN = [
    "CycleGAN",
    "Pix2Pix",
    "Conditional GAN (cGAN)",
    "StyleGAN",
    "SRGAN",
    "WGAN-GP",
]

_rng = _random.Random(NUMERO_DE_GRUPO)  # seed = número de grupo -> resultado reproducible
variante_asignada = _rng.choice(VARIANTES_GAN)
print(f"Grupo {NUMERO_DE_GRUPO} → variante asignada: {variante_asignada}")


### Qué tienen que preparar sobre su variante asignada (para la defensa)

1. **Problema que resuelve**: ¿qué limitación de la GAN "vanilla" (o de la DCGAN que armaron en esta notebook) motivó esta variante?
2. **Diferencias de arquitectura y/o loss**: ¿qué cambia respecto al Generador/Discriminador que implementaron acá? (por ejemplo: ¿usa dos generadores y dos discriminadores como CycleGAN? ¿condiciona la generación con una etiqueta o imagen, como cGAN/Pix2Pix? ¿agrega un término extra a la loss, como el gradient penalty de WGAN-GP?)
3. **Diagrama simple** del flujo de datos (inputs → generador(es) → discriminador(es) → losses). Puede ser una imagen insertada en esta notebook (celda de markdown con `![diagrama](ruta_a_imagen.png)`) o una descripción en un slide aparte para la defensa.
4. **Caso de uso real**: un ejemplo concreto de aplicación de esa variante (papers, productos, demos conocidas).
5. **Referencia**: el paper original (nombre, autores, año, link a arXiv si lo encuentran).

*(Espacio para completar la explicación y el diagrama de su variante asignada)*

---

**Referencias de los papers originales** (por si les toca buscar):
- CycleGAN: Zhu et al., 2017 — "Unpaired Image-to-Image Translation using Cycle-Consistent Adversarial Networks"
- Pix2Pix: Isola et al., 2017 — "Image-to-Image Translation with Conditional Adversarial Networks"
- Conditional GAN: Mirza & Osindero, 2014 — "Conditional Generative Adversarial Nets"
- StyleGAN: Karras et al., 2019 — "A Style-Based Generator Architecture for GANs"
- SRGAN: Ledig et al., 2017 — "Photo-Realistic Single Image Super-Resolution Using a GAN"
- WGAN-GP: Gulrajani et al., 2017 — "Improved Training of Wasserstein GANs"


## 7. Preguntas de reflexión (para la defensa)

1. ¿Qué diferencias notaron en la estabilidad del entrenamiento entre DCGAN y WGAN (mirando las curvas de loss)?
2. ¿Observaron mode collapse en alguna de las dos? ¿Cómo se dieron cuenta?
3. ¿Por qué WGAN necesita entrenar el crítico varias veces por cada paso del generador?
4. ¿Qué limitación tiene el *weight clipping* que motivó la aparición de WGAN-GP (gradient penalty)?
5. Sobre la variante de GAN que investigaron: ¿en qué se parece y en qué se diferencia fundamentalmente de la DCGAN que implementaron en esta notebook?
